In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from tinyconformal.classifier import BinaryMarginalConformalClassifier
from tinyshift.plot import efficiency_curve, reliability_curve, confusion_matrix;
import numpy as np
from pathlib import Path

In [ ]:
EXAMPLES_DIR = Path.cwd().parent
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))
from utils.plot_utils import histogram

In [ ]:
weights = [0.2, 0.8]

X, y = make_classification(
    n_samples=100000, 
    n_features=20, 
    n_informative=2,      
    weights=weights, 
    random_state=42,
    n_redundant=2)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_calib, y_train, y_calib = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

In [ ]:
rf = RandomForestClassifier(random_state=42, oob_score=True, n_jobs=-1, class_weight="balanced", max_depth=int(np.ceil(np.log2(len(X_train)) - 1)))
rf.fit(X_train, y_train)

In [ ]:
clf = BinaryMarginalConformalClassifier(rf)
clf.fit(y=y_train, oob=True)

In [ ]:
clf.calibrate(X_calib, y_calib)

In [ ]:
efficiency_curve(clf, X_test, y_test, fig_type="png")

In [ ]:
clf.evaluate(X_test, y_test, alpha=0.08)

In [ ]:
confusion_matrix(clf, X_test, y_test, fig_type="png")

Random Forest

In [ ]:
reliability_curve(clf.learner, X_test, y_test, "Random Forest", 15, "png")

In [ ]:
histogram(clf.learner, X_test, 15, "png")

Venn Abers

In [ ]:
reliability_curve(clf, X_test, y_test, "Venn Abers", 15, "png")

In [ ]:
histogram(clf, X_test, 15, "png")

Alpha - 0.05

In [ ]:
clf.evaluate(X_test, y_test, alpha=0.05)

In [ ]:
clf.alpha = 0.05
confusion_matrix(clf, X_test, y_test, fig_type="png")

Alpha - 0.10

In [ ]:
clf.evaluate(X_test, y_test, alpha=0.10)

In [ ]:
clf.alpha = 0.10
confusion_matrix(clf, X_test, y_test, fig_type="png")